# 08 — Postprocessing Workflow

With the transient region removed, we can now extract statistically meaningful
quantities from the steady-state window.  This notebook walks through:

1. Loading and trimming (recap)
2. Effective Sample Size (ESS)
3. Full statistics (mean, variance, confidence interval, …)
4. Cumulative statistics and additional-data estimation
5. Exporting results as a DataFrame or JSON
6. Diagnostic plots (trace, ACF, steady-state)
7. Ensemble postprocessing with all three aggregation techniques

> **Trim convention:** trimming always goes through the
> `QuantileTrimStrategy` / `TrimDataStreamOperation` objects — never
> `ds.trim()`.

---

## 1  Load and Trim

In [ ]:
import quends as qnds
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

# Load raw simulation output
ds = qnds.from_csv("gx/tprim_2_5_a.out.csv")

# Trim to steady state using the explicit strategy-operation pattern
strategy = QuantileTrimStrategy(window_size=20, robust=True)
op = TrimDataStreamOperation(strategy=strategy)
trimmed = op(ds, column_name="HeatFlux_st")

print(f"Raw steps    : {len(ds.data)}")
print(f"Trimmed steps: {len(trimmed.data)}")
print(f"SS t_start   : {trimmed.data['time'].iloc[0]:.3f}")

---

## 2  Effective Sample Size (ESS)

Autocorrelation in time-series data means consecutive samples are not
independent.  The ESS accounts for this, giving the number of *effectively*
independent draws — the quantity that actually determines statistical
uncertainty.

In [ ]:
ess = trimmed.effective_sample_size()
print(f"Effective Sample Size: {ess}")

---

## 3  Full Statistics

`compute_statistics()` returns a dict keyed by column name.  The `"sliding"`
method (default) uses a sliding-window estimator that is robust to
non-stationarity within the window.

In [ ]:
stats = trimmed.compute_statistics("HeatFlux_st")
print(stats)

### 3a  Mean and Confidence Interval separately

When you only need one of these quantities, the dedicated helpers avoid
re-computing unnecessary quantities.

In [ ]:
mu = trimmed.mean()
ci = trimmed.confidence_interval()

print(f"Mean               : {mu}")
print(f"Confidence interval: {ci}")

---

## 4  Cumulative Statistics

Cumulative statistics show how the estimated mean (and uncertainty) evolve as
more of the steady-state window is included.  They are the primary diagnostic
for assessing whether the simulation has run long enough.

In [ ]:
cumstats = trimmed.cumulative_statistics()
print(cumstats)

---

## 5  Additional Data Estimation

If the confidence interval is too wide, how much *more* data would you need?
`additional_data()` estimates the required additional simulation time given a
target reduction in uncertainty.  The `reduction_factor` is the fraction by
which you want to shrink the half-width of the CI.

In [ ]:
additional = trimmed.additional_data(reduction_factor=0.2)
print(f"Estimated additional simulation time needed: {additional}")

---

## 6  Exporting Results

`qnds.Exporter` converts computed statistics into display-ready formats.

* `display_dataframe()` — renders a `pandas.DataFrame` (ideal in Jupyter)
* `display_json()` — returns a JSON-serialisable dict / pretty-prints it

In [ ]:
exporter = qnds.Exporter()

# Render as a DataFrame
exporter.display_dataframe(stats["HeatFlux_st"])

In [ ]:
# Export as JSON
exporter.display_json(stats["HeatFlux_st"])

---

## 7  Diagnostic Plots

`qnds.Plotter` provides the standard diagnostics for plasma-physics UQ:

| Method | Purpose |
|--------|--------|
| `trace_plot(ds, cols)` | Raw time-series to visually assess the transient |
| `steady_state_automatic_plot(ds, cols)` | Overlay of automatic trim boundary |
| `plot_acf(ds)` | Autocorrelation function — needed to interpret ESS |

In [ ]:
plotter = qnds.Plotter()

# 1. Raw trace — always the first thing to look at
plotter.trace_plot(ds, ["HeatFlux_st"])

In [ ]:
# 2. Automatic steady-state detection overlaid on the trace
plotter.steady_state_automatic_plot(ds, ["HeatFlux_st"])

In [ ]:
# 3. Autocorrelation function of the trimmed series
plotter.plot_acf(trimmed)

---

## 8  Ensemble Postprocessing

`Ensemble.compute_statistics()` supports three aggregation **techniques**:

| Technique | Description |
|-----------|-------------|
| `0` | Concatenate all members, compute once |
| `1` | Average per-member statistics |
| `2` | Hierarchical / pooled estimator (recommended) |

Compute all three and compare to check robustness.

In [ ]:
files = [
    "gx/ensemble/tprim_2_5_a.out.csv",
    "gx/ensemble/tprim_2_5_b.out.csv",
]

ens = qnds.Ensemble([qnds.from_csv(f) for f in files])
trimmed_ens = ens.trim("HeatFlux_st", method="std", window_size=20)

for t in [0, 1, 2]:
    r = trimmed_ens.compute_statistics("HeatFlux_st", technique=t)
    print(f"Technique {t}: mean={r['mean']:.4f},  CI={r['confidence_interval']}")

---

## Summary

| Task | API |
|------|-----|
| Trim | `TrimDataStreamOperation(strategy)(ds, column_name=...)` |
| ESS | `trimmed.effective_sample_size()` |
| Stats | `trimmed.compute_statistics(col)` |
| Mean / CI | `trimmed.mean()` / `trimmed.confidence_interval()` |
| Cumulative | `trimmed.cumulative_statistics()` |
| More data? | `trimmed.additional_data(reduction_factor)` |
| Export | `qnds.Exporter().display_dataframe(...)` / `.display_json(...)` |
| Ensemble stats | `trimmed_ens.compute_statistics(col, technique=0|1|2)` |

**Next:** `09_end_to_end_pipeline.ipynb` — compose everything into a single
reproducible pipeline.